# 10 · Gold Layer - Star Schema DLT Pipeline
**Architecture:** Kimball Dimensional Model (Star Schema)

| Table | Type | SCD | PK | Silver Source |
|---|---|---|---|---|
| `dim_date` | Static Dimension | None (calendar) | `date_key` | - |
| `dim_car` | Streaming Dimension | **SCD Type 2** | `brand + model` | `car_catalog_transformation` |
| `dim_location` | Streaming Dimension | **SCD Type 2** | `city_prepositional` | `geography_transformation` |
| `dim_listing_details` | Streaming Dimension | **SCD Type 2** | `listing_id` | `listings_text_transformation` |
| `dim_listing_photos` | Streaming Dimension | **SCD Type 2** | `listing_id + photo_url_clean` | `listings_photo_transformation` |
| `fact_listings` | Fact (batch) | - | `listing_id` | `listings_silver_merged` |
| `agg_monthly_sales_trend` | Aggregate | - | - | `fact_listings` |
| `agg_brand_location_performance` | Aggregate | - | - | `fact_listings` |
| `agg_regional_market_depth` | Aggregate | - | - | `fact_listings` |
| `agg_comprehensive_kpi_cube` | Aggregate | - | - | `fact_listings` |
| `agg_top_10_brands_by_spend` | Aggregate | - | - | `fact_listings` |

**Why `dim_listing_photos`?**
`listings_photo_transformation` has 7,948,322 rows - one listing has MANY photos (one-to-many).
Modeling it as SCD2 dimension captures photo URL changes over time and satisfies the
**No Data Loss** requirement: all 7.9M Silver photo rows must appear in Gold.
`fact_listings` also carries `photo_count` (pre-aggregated) so dashboards don't need a join.


## Cell 1 · Imports & Configuration

In [0]:
import dlt
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "vstone_catalog"
SILVER  = f"{CATALOG}.silver"

GOLD_PROPS = {
    "quality"                   : "gold",
    "delta.enableChangeDataFeed": "true",
    "pipelines.reset.allowed"   : "true",
}


## Cell 2 · `dim_date` - Static Calendar Dimension
Gap-free date sequence 2010–2030. Includes **Year, Quarter, Month, Week** as required by the project spec. No SCD2 needed - dates are immutable.

In [0]:
# ── DIM_DATE - Static calendar dimension (no SCD2 needed: dates never change) ──
# Generates one row per day from 2010-01-01 to 2030-12-31.
# Includes Quarter and Week as required by the project spec.
# PK: date_key (DATE)

@dlt.table(
    name             = "dim_date",
    comment          = "Gold: Continuous gap-free Date dimension (2010–2030). "
                       "Static - generated via sequence(), no SCD2 required. "
                       "PK: date_key. Includes year, quarter, month, week, day.",
    table_properties = {**GOLD_PROPS, "pk": "date_key"},
)
def dim_date():
    return (
        spark.range(1)
        .selectExpr(
            "explode(sequence(to_date('2010-01-01'), to_date('2030-12-31'), interval 1 day)) as date_key"
        )
        .select(
            F.col("date_key"),
            F.year("date_key").alias("year"),
            F.quarter("date_key").alias("quarter"),          # Q1–Q4 (project requirement)
            F.month("date_key").alias("month"),
            F.date_format("date_key", "MMMM").alias("month_name"),
            F.weekofyear("date_key").alias("week_of_year"),  # ISO week (project requirement)
            F.dayofmonth("date_key").alias("day"),
            F.date_format("date_key", "EEEE").alias("day_name"),
            F.when(F.dayofweek("date_key").isin(1, 7), True)
             .otherwise(False).alias("is_weekend"),
        )
    )


## Cell 3 · `dim_car` - SCD Type 2
Tracks changes to car specifications (engine, generation, trim) over time. `apply_changes()` adds `__START_AT`, `__END_AT`, `__CURRENT` metadata. Source: `silver.car_catalog_transformation`.

In [0]:
# ── DIM_CAR - SCD Type 2 (brand + model is the natural key) ──────────────────
# Source: silver.car_catalog_transformation
# SCD2 tracks changes to engine specs, trim, generation over time.
# apply_changes() adds __START_AT, __END_AT, __CURRENT metadata columns.
# PK: brand + model (composite)

dlt.create_streaming_table(
    name             = "dim_car",
    comment          = "Gold SCD2: Car specifications dimension. Tracks changes to engine, "
                       "generation, and trim over time. PK: brand+model (composite). "
                       "Source: silver.car_catalog_transformation.",
    table_properties = {**GOLD_PROPS, "scd_type": "2", "pk": "brand,model"},
)

@dlt.view(name="car_source_v")
def car_source_v():
    # skipChangeCommits=true: skip UPDATE/DELETE commits from Silver
    # (only STREAMING UPDATE / APPEND rows are meaningful for dimension loading)
    return (
        spark.readStream
        .option("skipChangeCommits", "true")
        .table(f"{SILVER}.car_catalog_transformation")
        .select(
            "brand",
            "model",
            "generation",
            "trim_level",
            "engine_volume_l",
            "engine_power_hp",
            "fuel_type",
            "transmission",
            "drive_type",
            "body_type",
            "seats_count",
            "max_speed_kmh",
            "silver_load_dt",          # sequence_by column for SCD2 ordering
        )
    )

dlt.apply_changes(
    target           = "dim_car",
    source           = "car_source_v",
    keys             = ["brand", "model"],
    sequence_by      = F.col("silver_load_dt"),
    stored_as_scd_type = 2,
)


## Cell 4 · `dim_location` - SCD Type 2
Tracks changes to city coordinates and canonical names over time. Source: `silver.geography_transformation`.

In [0]:
# ── DIM_LOCATION - SCD Type 2 (city_prepositional is the natural key) ─────────
# Source: silver.geography_transformation
# SCD2 tracks if a city's coordinates or canonical name change over time.
# PK: city_prepositional

dlt.create_streaming_table(
    name             = "dim_location",
    comment          = "Gold SCD2: City/region dimension with lat/lon coordinates. "
                       "Tracks changes to city_name and coordinates over time. "
                       "PK: city_prepositional. Source: silver.geography_transformation.",
    table_properties = {**GOLD_PROPS, "scd_type": "2", "pk": "city_prepositional"},
)

@dlt.view(name="location_source_v")
def location_source_v():
    return (
        spark.readStream
        .option("skipChangeCommits", "true")
        .table(f"{SILVER}.geography_transformation")
        .select(
            "city_prepositional",
            "city_name",
            "latitude",
            "longitude",
            "silver_load_dt",
        )
    )

dlt.apply_changes(
    target             = "dim_location",
    source             = "location_source_v",
    keys               = ["city_prepositional"],
    sequence_by        = F.col("silver_load_dt"),
    stored_as_scd_type = 2,
)


## Cell 5 · `dim_listing_details` - SCD Type 2
Tracks edits to listing description text over time. Source: `silver.listings_text_transformation`.

In [0]:
# ── DIM_LISTING_DETAILS - SCD Type 2 (listing_id is the natural key) ──────────
# Source: silver.listings_text_transformation
# SCD2 tracks if a listing's description text is edited over time.
# PK: listing_id

dlt.create_streaming_table(
    name             = "dim_listing_details",
    comment          = "Gold SCD2: Listing text description dimension. "
                       "Tracks edits to listing description over time. "
                       "PK: listing_id. Source: silver.listings_text_transformation.",
    table_properties = {**GOLD_PROPS, "scd_type": "2", "pk": "listing_id"},
)

@dlt.view(name="text_source_v")
def text_source_v():
    return (
        spark.readStream
        .option("skipChangeCommits", "true")
        .table(f"{SILVER}.listings_text_transformation")
        .select(
            "listing_id",
            F.col("text").alias("description_clean"),
            "silver_load_dt",
        )
    )

dlt.apply_changes(
    target             = "dim_listing_details",
    source             = "text_source_v",
    keys               = ["listing_id"],
    sequence_by        = F.col("silver_load_dt"),
    stored_as_scd_type = 2,
)


## Cell 6 · `dim_listing_photos` - SCD Type 2
**This is the 5th Silver table (`listings_photo_transformation`) brought into Gold.**

7,948,322 rows - one listing has many photos (one-to-many). Composite natural key: `listing_id + photo_url_clean`. SCD2 tracks photo URL changes (CDN migrations, photo replacements). `fact_listings` also carries `photo_count` (pre-aggregated) for dashboards that don't need to traverse this dimension.

In [0]:
# ── DIM_LISTING_PHOTOS - SCD Type 2 (listing_id + photo_url_clean is the natural key) ─
# Source: silver.listings_photo_transformation  (7,948,322 rows)
# One listing can have MANY photos - this is a one-to-many dimension.
# SCD2 tracks photo URL changes over time (e.g. CDN migration, photo replacement).
# Composite natural key: listing_id + photo_url_clean (the Silver dedup key).
# PK: listing_id + photo_url_clean

dlt.create_streaming_table(
    name             = "dim_listing_photos",
    comment          = "Gold SCD2: Photo URLs per car listing. Tracks photo changes over time. "
                       "7.9M rows - one listing has many photos (one-to-many). "
                       "PK: listing_id + photo_url_clean. Source: silver.listings_photo_transformation.",
    table_properties = {**GOLD_PROPS, "scd_type": "2", "pk": "listing_id,photo_url_clean"},
)

@dlt.view(name="photo_source_v")
def photo_source_v():
    return (
        spark.readStream
        .option("skipChangeCommits", "true")
        .table(f"{SILVER}.listings_photo_transformation")
        .select(
            "listing_id",
            "photo_url",           # original URL (raw)
            "photo_url_clean",     # standardized URL (dedup key from Silver)
            "silver_load_dt",      # sequence_by for SCD2 ordering
        )
    )

dlt.apply_changes(
    target             = "dim_listing_photos",
    source             = "photo_source_v",
    keys               = ["listing_id", "photo_url_clean"],  # composite natural key
    sequence_by        = F.col("silver_load_dt"),
    stored_as_scd_type = 2,
)


## Cell 7 · `fact_listings` - Central Fact Table
Batch read from `silver.listings_silver_merged` + left-join `photo_count` from `listings_photo_transformation`. **No additional dedup or filter on the main listing rows** - Silver is the dedup boundary. Gold adds: `car_age_at_listing`, `is_high_mileage`, `price_per_hp_usd`, `photo_count`.

In [0]:
# ── FACT_LISTINGS - Central fact table (batch read from Silver) ───────────────
# Source: silver.listings_silver_merged  (1,083,237 rows, already deduped)
# No row drops allowed: Silver -> Gold must be 1:1 (no additional dedup/filter).
#
# FK relationships:
#   listing_date  -> dim_date.date_key        (DATE join)
#   brand + model -> dim_car.(brand, model)   (composite FK)
#   location_key  -> dim_location.city_prepositional
#   listing_id    -> dim_listing_details.listing_id
#
# Derived metrics (all computed in Silver, preserved here 1:1):
#   car_age_years       = 2023 - manufacture_year   (from Silver)
#   price_usd           = round(price_rub / 82.5, 2) (from Silver)
#   price_category      = BUDGET/MID_RANGE/PREMIUM/LUXURY (from Silver)
#
# Additional Gold-level derived columns:
#   car_age_at_listing  = listing_year - manufacture_year
#   is_high_mileage     = mileage_km > 100,000
#   price_per_hp_usd    = price_usd / engine_power

@dlt.table(
    name             = "fact_listings",
    comment          = "Gold Fact: Car listings with full PK/FK star schema relationships. "
                       "No additional dedup - Silver is the dedup boundary. "
                       "PK: listing_id. Derived: car_age_at_listing, is_high_mileage, price_per_hp_usd.",
    table_properties = {
        **GOLD_PROPS,
        "type"        : "fact",
        "pk"          : "listing_id",
        "fk_date"     : "listing_date -> dim_date.date_key",
        "fk_car"      : "brand+model -> dim_car.(brand,model)",
        "fk_location" : "location_key -> dim_location.city_prepositional",
        "fk_details"  : "listing_id -> dim_listing_details.listing_id",
    },
)
def fact_listings():
    df = spark.table(f"{SILVER}.listings_silver_merged")

    # Pre-aggregate photo count per listing from the photo Silver table.
    # This avoids a join at query time on every dashboard query, and ensures
    # photo data is represented in the fact table even for dashboards that
    # don't want to join through dim_listing_photos.
    photo_counts = (
        spark.table(f"{SILVER}.listings_photo_transformation")
        .groupBy("listing_id")
        .agg(F.count("photo_url_clean").alias("photo_count"))
    )

    return (
        df
        .join(photo_counts, on="listing_id", how="left")   # left join: listings without photos get NULL
        .withColumn("photo_count", F.coalesce(F.col("photo_count"), F.lit(0)))
        .select(
        # ── Primary Key ───────────────────────────────────────────────────────
        "listing_id",

        # ── Foreign Keys (dimension linkage) ─────────────────────────────────
        F.col("listing_date").cast("date").alias("listing_date"),   # FK -> dim_date.date_key
        "brand",                                                     # FK -> dim_car (composite)
        "model",                                                     # FK -> dim_car (composite)
        F.col("city_prepositional").alias("location_key"),           # FK -> dim_location

        # ── Listing attributes ────────────────────────────────────────────────
        "manufacture_year",
        "engine_power",
        "mileage_km",
        "fuel_type",
        "transmission_type",
        "drive_type",
        "steering_wheel",
        "trim_level",
        "has_license",
        "color_r",
        "color_g",
        "color_b",

        # ── Silver-computed financials (preserved 1:1, no recalculation) ─────
        "price_rub",
        "price_usd",
        "price_category",
        "car_age_years",        # 2023 - manufacture_year (Silver)
        "listing_year",
        "listing_month",

        # ── Gold-level derived metrics ────────────────────────────────────────
        (F.year(F.col("listing_date")) - F.col("manufacture_year"))
            .alias("car_age_at_listing"),
        F.when(F.col("mileage_km") > 100000, True)
            .otherwise(False).alias("is_high_mileage"),
        F.round(F.col("price_usd") / F.nullif(F.col("engine_power"), F.lit(0)), 2)
            .alias("price_per_hp_usd"),
        "photo_count",             # pre-joined from listings_photo_transformation

        # ── Audit ─────────────────────────────────────────────────────────────
        "bronze_load_dt",
        "bronze_source_file",
        "silver_load_dt",
        F.current_timestamp().alias("gold_load_dt"),
        )    # close .select()
    )        # close .join() chain


## Cell 8 · Aggregates (5 tables)
All aggregates read from `fact_listings` via `dlt.read()`. Every table includes `gold_load_dt` as an audit column.

In [0]:
# ── AGGREGATES ────────────────────────────────────────────────────────────────
# All aggregates read from fact_listings via dlt.read() (same pipeline).
# gold_load_dt added to every aggregate as audit column.

# ── AGG 1: Monthly Sales Trend by Brand ──────────────────────────────────────
@dlt.table(
    name             = "agg_monthly_sales_trend",
    comment          = "Gold Aggregate: Listing volume and revenue trend by month, brand, "
                       "and price category. Enables time-series dashboards.",
    table_properties = {**GOLD_PROPS, "type": "aggregate"},
)
def agg_monthly_trend():
    return (
        dlt.read("fact_listings")
        .withColumn("month_year", F.date_format(F.col("listing_date"), "yyyy-MM"))
        .groupBy("month_year", "brand", "price_category")
        .agg(
            F.count("listing_id").alias("total_listings"),
            F.round(F.avg("price_rub"), 0).alias("avg_price_rub"),
            F.round(F.avg("price_usd"), 0).alias("avg_price_usd"),
            F.round(F.sum("price_usd"), 0).alias("total_revenue_usd"),
        )
        .orderBy("month_year", F.desc("total_listings"))
        .withColumn("gold_load_dt", F.current_timestamp())
    )

# ── AGG 2: Brand + Location Performance ──────────────────────────────────────
@dlt.table(
    name             = "agg_brand_location_performance",
    comment          = "Gold Aggregate: Brand performance broken down by region. "
                       "Enables geo-filtered brand comparison dashboards.",
    table_properties = {**GOLD_PROPS, "type": "aggregate"},
)
def agg_brand_performance():
    return (
        dlt.read("fact_listings")
        .groupBy("brand", "location_key")
        .agg(
            F.count("listing_id").alias("listing_count"),
            F.round(F.avg("price_usd"), 0).alias("avg_price_usd"),
            F.round(F.avg("mileage_km"), 0).alias("avg_mileage_km"),
            F.max("engine_power").alias("max_hp_in_region"),
        )
        .withColumn("gold_load_dt", F.current_timestamp())
    )

# ── AGG 3: Regional Market Depth ─────────────────────────────────────────────
@dlt.table(
    name             = "agg_regional_market_depth",
    comment          = "Gold Aggregate: Inventory depth by city, fuel type, and price segment. "
                       "Enables regional supply/demand analysis.",
    table_properties = {**GOLD_PROPS, "type": "aggregate"},
)
def agg_regional_depth():
    return (
        dlt.read("fact_listings")
        .groupBy("location_key", "fuel_type", "price_category")
        .agg(
            F.count("listing_id").alias("inventory_count"),
            F.round(F.avg("mileage_km"), 0).alias("avg_mileage"),
            F.round(F.avg("price_usd"), 0).alias("avg_price_usd"),
        )
        .withColumn("gold_load_dt", F.current_timestamp())
    )

# ── AGG 4: Comprehensive KPI Cube ─────────────────────────────────────────────
@dlt.table(
    name             = "agg_comprehensive_kpi_cube",
    comment          = "Gold Aggregate: Multi-dimensional KPI cube for flexible BI dashboarding. "
                       "Supports slicing by brand, model, year, segment, fuel, mileage flag.",
    table_properties = {**GOLD_PROPS, "type": "aggregate"},
)
def agg_kpi_cube():
    return (
        dlt.read("fact_listings")
        .groupBy("brand", "model", "manufacture_year", "price_category", "fuel_type", "is_high_mileage")
        .agg(
            F.count("listing_id").alias("listing_volume"),
            F.round(F.avg("price_usd"), 2).alias("avg_market_price_usd"),
            F.round(F.avg("car_age_at_listing"), 1).alias("avg_age_at_listing"),
            F.round(F.avg("mileage_km"), 0).alias("avg_mileage_km"),
        )
        .withColumn("gold_load_dt", F.current_timestamp())
    )

# ── AGG 5: Top 10 Brands by Total Market Value ───────────────────────────────
@dlt.table(
    name             = "agg_top_10_brands_by_spend",
    comment          = "Gold Aggregate: Top 10 car brands ranked by cumulative market value (USD). "
                       "Fulfils 'Top 10 by spend' dashboard requirement.",
    table_properties = {**GOLD_PROPS, "type": "aggregate"},
)
def agg_top_10_brands():
    return (
        dlt.read("fact_listings")
        .groupBy("brand")
        .agg(
            F.round(F.sum("price_usd"), 0).alias("total_market_value_usd"),
            F.count("listing_id").alias("total_listings"),
            F.round(F.avg("price_usd"), 0).alias("avg_price_usd"),
        )
        .orderBy(F.desc("total_market_value_usd"))
        .limit(10)
        .withColumn("gold_load_dt", F.current_timestamp())
    )
